# Incident 3 — The Concurrent Fleet Overspend

**What happened:** A LangGraph supervisor spawned 10 research sub-agents simultaneously. Each read the same $1,000 available balance. All 10 saw enough funds. All 10 were approved. Total spend: **$2,000**. Budget exceeded by $1,000.

**Why it happens:** Classic TOCTOU (time-of-check to time-of-use) race. The agents read the balance concurrently — all see $1,000 available — then all write charges concurrently. Without atomic read-modify-write, every agent passes the check before any charge lands.

**The fix:** FiGuard uses SERIALIZABLE isolation on every `authorize()` write. A pessimistic write lock on the budget row makes each agent's read-modify-write atomic. Exactly 5 agents are authorized ($1,000), the remaining 5 are denied. Budget never exceeded.

In [ ]:
import sys
!pip install "figuard>=0.3.2" --upgrade -q
for _mod in list(sys.modules.keys()):
    if _mod.startswith("figuard"):
        del sys.modules[_mod]
import figuard
print(f"✓ FiGuard {figuard.__version__} ready")

## Without FiGuard — all 10 agents approved, $2,000 charged

Each thread reads the same balance and approves itself. The budget check is not atomic.

In [ ]:
# WITHOUT FiGuard — no atomic check, TOCTOU race

import threading

budget_limit = 1000.00
available = budget_limit  # shared state — no locking
results = []
print_lock = threading.Lock()


def agent_spend_unsafe(agent_id):
    global available
    amount = 200.00
    # Read — all agents see $1,000 available at the same time
    if available >= amount:
        available -= amount   # write races with other threads
        decision = "AUTHORIZED"
    else:
        decision = "DENIED"
    with print_lock:
        results.append((agent_id, decision))
        print(f"Agent {agent_id:2d}: {decision}  $200.00")


print(f"Budget: ${budget_limit:.2f}  |  10 agents × $200 = $2,000 requested")
print()

threads = [threading.Thread(target=agent_spend_unsafe, args=(i,)) for i in range(10)]
for t in threads:
    t.start()
for t in threads:
    t.join()

authorized = [r for r in results if r[1] == "AUTHORIZED"]
total_charged = len(authorized) * 200.00
print()
print(f"Authorized: {len(authorized)}/10  |  Total charged: ${total_charged:.2f}")
print(f"Budget exceeded: {total_charged > budget_limit}  (expected ≤ ${budget_limit:.2f}, got ${total_charged:.2f})")

## With FiGuard — serializable isolation, exactly 5 authorized

FiGuard locks the budget row on every `authorize()`. Each agent's read-modify-write is atomic. Exactly 5 agents ($1,000) are authorized; the remaining 5 are denied with `BUDGET_EXHAUSTED`.

In [ ]:
import threading
from figuard import FiGuardClient

client = FiGuardClient(
    api_key="sb_live_demo",
    base_url="https://figuard-sandbox-1.onrender.com",
)

budget = client.create_budget(
    user_id="supervisor",
    total_limit=1000.00,
    currency="USD",
    expires_in="1h",
)

# Extract session token from tokens array (one entry per dimension)
tokens = {t.category: t.session_token for t in budget.tokens}
session_token = tokens["default"]  # simple budget uses "default" category


results = []
lock = threading.Lock()


def agent_spend(agent_id):
    auth = client.authorize(
        session_token=session_token,
        agent_id=f"research_agent_{agent_id}",
        action_type="COMPUTE",
        description=f"Research subtask {agent_id}",
        requested_quantity=200.00,
        idempotency_key=f"research-task-{agent_id}",
    )
    with lock:
        results.append((agent_id, auth.decision,
                        auth.denial_reason if not auth.is_authorized else None))
        status = "✓ AUTHORIZED" if auth.is_authorized else "✗ DENIED    "
        reason = f"[{auth.denial_reason}]" if not auth.is_authorized else ""
        print(f"Agent {agent_id:2d}: {status}  $200.00  {reason}")


print(f"Budget: ${budget.total_limit:.2f}  |  10 agents × $200 = $2,000 requested")
print()

threads = [threading.Thread(target=agent_spend, args=(i,)) for i in range(10)]
for t in threads:
    t.start()
for t in threads:
    t.join()

authorized = [r for r in results if r[1] == "AUTHORIZED"]
total = len(authorized) * 200.0

print()
print(f"✓ Authorized: {len(authorized)}/10 agents  (${total:.2f} of ${budget.total_limit:.2f})")
print(f"  Budget never exceeded: {total <= budget.total_limit}")

print(f"\n→ See the ledger: https://figuard-sandbox-1.onrender.com/ui")